# Notebook 03 — Distribution Shift Robustness

Tests how selective prediction strategies respond to three types of distribution shift:
1. Gaussian noise injection
2. Feature perturbation (covariate shift)
3. Class imbalance shift


In [1]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.preprocessing import (
    load_raw_dataset, engineer_features, split_and_scale,
    inject_noise, perturb_features, imbalance_shift
)
from src.training import build_model, train_model
from src.selective_prediction import (
    static_threshold_selection, cost_aware_selection, dynamic_threshold_selection
)
from src.evaluation import evaluate_strategy
from src.visualization import plot_distribution_shift_comparison
from src.utils import set_seed
set_seed(42)
print('Ready')

Ready


## 1. Setup

In [2]:
df = load_raw_dataset()
df = engineer_features(df)
splits = split_and_scale(df)

model = build_model('rf')
fitted = train_model(model, splits['X_train'], splits['y_train'],
                     splits['X_val'], splits['y_val'], calibrate=True)
X_test = splits['X_test']
y_test = splits['y_test']
print('RF calibrated and ready')

[12:25:50] WARNING | src.preprocessing | Real dataset not found. Generating synthetic fallback (20 000 samples, 5 %% fraud rate).
[12:25:50] INFO | src.training | Training RandomForestClassifier …
[12:25:53] INFO | src.training | Applying probability calibration (isotonic) ...


RF calibrated and ready


## 2. Gaussian Noise Injection

In [3]:
noise_levels = [0.0, 0.25, 0.5, 1.0, 2.0]
noise_shift = {}

for noise in noise_levels:
    X_sh = inject_noise(X_test, noise_std=noise, seed=42)
    row = {}
    for strat, fn in [
        ('static',     lambda m,X: static_threshold_selection(m, X, threshold=0.70)),
        ('cost_aware', lambda m,X: cost_aware_selection(m, X, cost_fp=1, cost_fn=5, cost_abstain=0.5)),
        ('dynamic',    lambda m,X: dynamic_threshold_selection(m, X, target_coverage=0.80)),
    ]:
        res = fn(fitted, X_sh)
        row[strat] = evaluate_strategy(res, y_test)
    noise_shift[f'noise={noise}'] = row

# Print table
print(f'{"Condition":<15} {"Strategy":<14} {"Coverage":>10} {"Sel.Risk":>10}')
print('-'*52)
for cond, strategies in noise_shift.items():
    for strat, metrics in strategies.items():
        print(f'{cond:<15} {strat:<14} {metrics["coverage"]:>10.3f} {metrics["selective_risk"]:>10.4f}')

Condition       Strategy         Coverage   Sel.Risk
----------------------------------------------------
noise=0.0       static              0.964     0.0280
noise=0.0       cost_aware          0.904     0.0196
noise=0.0       dynamic             0.801     0.0087
noise=0.25      static              0.958     0.0279
noise=0.25      cost_aware          0.897     0.0234
noise=0.25      dynamic             0.800     0.0109
noise=0.5       static              0.945     0.0291
noise=0.5       cost_aware          0.861     0.0311
noise=0.5       dynamic             0.800     0.0141
noise=1.0       static              0.900     0.0400
noise=1.0       cost_aware          0.745     0.0550
noise=1.0       dynamic             0.800     0.0244
noise=2.0       static              0.829     0.0642
noise=2.0       cost_aware          0.517     0.1465
noise=2.0       dynamic             0.801     0.0606


## 3. Imbalance Shift

In [4]:
imb_ratios = [0.05, 0.10, 0.15, 0.20]
imb_shift = {}

for ratio in imb_ratios:
    X_sh, y_sh = imbalance_shift(X_test, y_test, target_fraud_ratio=ratio, seed=42)
    row = {}
    for strat, fn in [
        ('static',     lambda m,X: static_threshold_selection(m, X, threshold=0.70)),
        ('cost_aware', lambda m,X: cost_aware_selection(m, X, cost_fp=1, cost_fn=5, cost_abstain=0.5)),
        ('dynamic',    lambda m,X: dynamic_threshold_selection(m, X, target_coverage=0.80)),
    ]:
        res = fn(fitted, X_sh)
        row[strat] = evaluate_strategy(res, y_sh)
    imb_shift[f'fraud={ratio:.0%}'] = row

print(f'{"Fraud Rate":<14} {"Strategy":<14} {"Coverage":>10} {"Sel.Risk":>10}')
print('-'*52)
for cond, strategies in imb_shift.items():
    for strat, metrics in strategies.items():
        print(f'{cond:<14} {strat:<14} {metrics["coverage"]:>10.3f} {metrics["selective_risk"]:>10.4f}')

Fraud Rate     Strategy         Coverage   Sel.Risk
----------------------------------------------------
fraud=5%       static              0.964     0.0280
fraud=5%       cost_aware          0.904     0.0196
fraud=5%       dynamic             0.801     0.0087
fraud=10%      static              0.948     0.0535
fraud=10%      cost_aware          0.887     0.0342
fraud=10%      dynamic             0.800     0.0201
fraud=15%      static              0.932     0.0818
fraud=15%      cost_aware          0.873     0.0500
fraud=15%      dynamic             0.800     0.0442
fraud=20%      static              0.916     0.1087
fraud=20%      cost_aware          0.859     0.0647
fraud=20%      dynamic             0.802     0.0680


## 4. Combined Shift Plots

In [5]:
combined = {**noise_shift, **imb_shift}
path = plot_distribution_shift_comparison(combined, metric_key='selective_risk', model_name='rf')
img = plt.imread(str(path))
fig, ax = plt.subplots(figsize=(11,5))
ax.imshow(img); ax.axis('off'); plt.show()

path2 = plot_distribution_shift_comparison(combined, metric_key='coverage', model_name='rf')
img2 = plt.imread(str(path2))
fig2, ax2 = plt.subplots(figsize=(11,5))
ax2.imshow(img2); ax2.axis('off'); plt.show()

## 5. Key Insights

- **Noise injection**: Static thresholds degrade predictably — coverage drops as noise increases.
  Dynamic thresholding is most stable because it recalibrates to the shifted distribution.
- **Imbalance shift**: Cost-aware abstention adapts best to higher fraud rates because it
  directly incorporates the asymmetric FN penalty (cost_fn=5).
- **General finding**: No single strategy dominates all shift types, reinforcing the value
  of the tri-action framework that can combine elements of each.
